# Breast Cancer Detection — Notebook 5
## Swin Transformer on CBIS-DDSM Mammography

Fine-tunes a **Swin-Tiny** transformer (via `timm`) on full mammograms from **CBIS-DDSM** (curated DDSM, mass + calcification cases). Labels come from the pathology column (**MALIGNANT = 1** vs BENIGN / BENIGN_WITHOUT_CALLBACK = 0) and the dataset's official train/test split is respected. Trains with mixed precision (AMP) to fit Colab's T4.

### 1. Setup

In [ ]:
!pip install -q timm

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

base_path = "/content/drive/MyDrive/breast_cancer_project"

for folder in ["data", "models", "results", "plots"]:
    os.makedirs(os.path.join(base_path, folder), exist_ok=True)

print("Project folders ready")

In [ ]:
from google.colab import files
files.upload()   # upload your kaggle.json here

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

### 2. Dataset — CBIS-DDSM (JPEG release)

Note: this download is ~6 GB.

In [ ]:
!kaggle datasets download -d awsaf49/cbis-ddsm-breast-cancer-image-dataset
!unzip -q cbis-ddsm-breast-cancer-image-dataset.zip -d data

In [ ]:
!ls data
!ls data/csv

In [ ]:
import pandas as pd
import os

dicom = pd.read_csv("data/csv/dicom_info.csv")
print(dicom["SeriesDescription"].value_counts())

# map: series UID folder -> path of the full-mammogram JPEG on disk
full_mammo = dicom[dicom["SeriesDescription"] == "full mammogram images"]

full_mammo_paths = {}
for p in full_mammo["image_path"]:
    uid = p.split("/")[-2]
    full_mammo_paths[uid] = p.replace("CBIS-DDSM/jpeg", "data/jpeg")

print("Full mammograms:", len(full_mammo_paths))

In [ ]:
mass_train = pd.read_csv("data/csv/mass_case_description_train_set.csv")
mass_test = pd.read_csv("data/csv/mass_case_description_test_set.csv")
calc_train = pd.read_csv("data/csv/calc_case_description_train_set.csv")
calc_test = pd.read_csv("data/csv/calc_case_description_test_set.csv")

print("mass train/test:", len(mass_train), len(mass_test))
print("calc train/test:", len(calc_train), len(calc_test))
print(mass_train["pathology"].value_counts())

In [ ]:
def build_df(case_df):
    rows = []
    for _, r in case_df.iterrows():
        uid = r["image file path"].split("/")[-2]
        path = full_mammo_paths.get(uid)
        if path is not None and os.path.exists(path):
            rows.append({
                "path": path,
                "label": 1 if r["pathology"] == "MALIGNANT" else 0,
                "pathology": r["pathology"],
                "abnormality": r["abnormality type"],
            })
    return pd.DataFrame(rows).drop_duplicates(subset="path").reset_index(drop=True)

train_full_df = build_df(pd.concat([mass_train, calc_train], ignore_index=True))
test_df = build_df(pd.concat([mass_test, calc_test], ignore_index=True))

print("Train (official):", len(train_full_df), "| Test (official):", len(test_df))
print(train_full_df["label"].value_counts())

In [ ]:
import torch
import random
import numpy as np

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

In [ ]:
from sklearn.model_selection import train_test_split

# carve a validation split out of the official training set
train_df, val_df = train_test_split(train_full_df, test_size=0.15,
                                    stratify=train_full_df["label"],
                                    random_state=42)

print("Train:", len(train_df), "| Val:", len(val_df), "| Test:", len(test_df))

In [ ]:
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(7),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

In [ ]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image

class MammogramDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["path"]).convert("RGB")   # grayscale -> 3 channels
        label = row["label"]

        if self.transform:
            image = self.transform(image)

        return image, label

train_loader = DataLoader(MammogramDataset(train_df, train_transform),
                          batch_size=16, shuffle=True, num_workers=2)
val_loader = DataLoader(MammogramDataset(val_df, val_transform),
                        batch_size=16, num_workers=2)
test_loader = DataLoader(MammogramDataset(test_df, val_transform),
                         batch_size=16, num_workers=2)

img, label = MammogramDataset(train_df, train_transform)[0]
print(img.shape, label)

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

sample = pd.concat([train_df[train_df.label == 0].head(4),
                    train_df[train_df.label == 1].head(4)]).reset_index(drop=True)

fig, axes = plt.subplots(2, 4, figsize=(12, 7))
for ax, (_, row) in zip(axes.flat, sample.iterrows()):
    ax.imshow(Image.open(row["path"]).convert("L"), cmap="gray")
    ax.set_title(f'{row["pathology"]}\n({row["abnormality"]})', fontsize=9)
    ax.axis("off")

plt.tight_layout()
plt.savefig(base_path + "/plots/cbis_ddsm_samples.png", dpi=150, bbox_inches="tight")
plt.show()

### 3. Model — Swin-Tiny (`timm`) + AMP

In [ ]:
import timm
import torch.nn as nn

model = timm.create_model("swin_tiny_patch4_window7_224",
                          pretrained=True, num_classes=1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("Trainable parameters:",
      sum(p.numel() for p in model.parameters() if p.requires_grad))

In [ ]:
n_pos = int(train_df["label"].sum())
n_neg = len(train_df) - n_pos
pos_weight = torch.tensor([n_neg / n_pos], device=device)
print("pos_weight:", round(pos_weight.item(), 3))

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-5, weight_decay=0.05)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

scaler = torch.cuda.amp.GradScaler()   # mixed precision

### 4. Training (mixed precision)

In [ ]:
EPOCHS = 10
history = {"train_loss": [], "val_loss": [], "val_acc": []}
best_val_acc = 0.0

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for i, (images, labels) in enumerate(train_loader):
        images = images.to(device)
        labels = labels.float().to(device)

        with torch.cuda.amp.autocast():
            outputs = model(images).squeeze(1)
            loss = criterion(outputs, labels)

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

        if i % 50 == 0:
            print(f"Epoch {epoch+1}, Step {i}, Loss: {loss.item():.4f}")
    scheduler.step()

    # ---------------- validation ----------------
    model.eval()
    val_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.float().to(device)

            outputs = model(images).squeeze(1)
            val_loss += criterion(outputs, labels).item()

            preds = (torch.sigmoid(outputs) > 0.5).int()
            correct += (preds == labels.int()).sum().item()
            total += labels.size(0)

    train_loss = total_loss / len(train_loader)
    val_loss = val_loss / len(val_loader)
    val_acc = correct / total

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), base_path + "/models/swin_cbis_ddsm_best.pth")
        print("  -> new best model saved")

    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

In [ ]:
import matplotlib.pyplot as plt

epochs_range = range(1, len(history["train_loss"]) + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs_range, history["train_loss"], color="#0072B2", lw=2, label="Train loss")
axes[0].plot(epochs_range, history["val_loss"], color="#D55E00", lw=2, label="Val loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Loss curves")
axes[0].legend()

axes[1].plot(epochs_range, history["val_acc"], color="#009E73", lw=2, label="Val accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Validation accuracy")
axes[1].legend()

plt.tight_layout()
plt.savefig(base_path + "/plots/swin_cbis_ddsm_training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

### 5. Evaluation on the official CBIS-DDSM test split

In [ ]:
from sklearn.metrics import (confusion_matrix, classification_report,
                             roc_auc_score, accuracy_score)

model.load_state_dict(torch.load(base_path + "/models/swin_cbis_ddsm_best.pth"))
model.eval()

all_probs, all_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images).squeeze(1)
        probs = torch.sigmoid(outputs)

        all_probs.extend(probs.cpu().numpy())
        all_labels.extend(np.array(labels))

all_probs = np.array(all_probs)
all_labels = np.array(all_labels)
preds = (all_probs > 0.5).astype(int)

acc = accuracy_score(all_labels, preds)
auc = roc_auc_score(all_labels, all_probs)

print("Test Accuracy:", round(acc, 4))
print("ROC-AUC:", round(auc, 4))
print("\nConfusion Matrix:\n", confusion_matrix(all_labels, preds))
print("\nClassification Report:\n")
print(classification_report(all_labels, preds,
                            target_names=["benign", "malignant"]))

In [ ]:
import seaborn as sns
from sklearn.metrics import roc_curve

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Confusion matrix (single-hue sequential colormap)
cm = confusion_matrix(all_labels, preds)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=["benign", "malignant"],
            yticklabels=["benign", "malignant"], ax=axes[0])
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Actual")
axes[0].set_title("Confusion Matrix")

# ROC curve
fpr, tpr, _ = roc_curve(all_labels, all_probs)
axes[1].plot(fpr, tpr, color="#0072B2", lw=2, label=f"ROC (AUC = {auc:.3f})")
axes[1].plot([0, 1], [0, 1], color="gray", lw=1, linestyle="--", label="Chance")
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("ROC Curve")
axes[1].legend()

plt.tight_layout()
plt.savefig(base_path + "/plots/swin_cbis_ddsm_confusion_roc.png", dpi=150, bbox_inches="tight")
plt.show()

### 6. Save artifacts to Drive

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

precision = precision_score(all_labels, preds)
recall = recall_score(all_labels, preds)
f1 = f1_score(all_labels, preds)

results_text = f"""MODEL: Swin Transformer (Swin-Tiny)
DATASET: CBIS-DDSM full mammograms (official test split)
LOSS: BCEWithLogitsLoss (pos_weight from data)

Test Accuracy: {acc:.4f}
ROC-AUC: {auc:.4f}

Malignant class:
Precision: {precision:.4f}
Recall: {recall:.4f}
F1-score: {f1:.4f}

Confusion Matrix:
{confusion_matrix(all_labels, preds)}

Classification Report:
{classification_report(all_labels, preds)}
"""

with open(base_path + "/results/swin_cbis_ddsm_results.txt", "w") as f:
    f.write(results_text)

print(results_text)
print("Results saved")

In [ ]:
results_df = pd.DataFrame({
    "path": test_df["path"].values,
    "true_label": all_labels,
    "predicted_label": preds,
    "probability": all_probs
})

results_df.to_csv(base_path + "/results/swin_cbis_ddsm_predictions.csv", index=False)
print("Predictions saved")

In [ ]:
import json

config = {
    "model": "swin_tiny_patch4_window7_224 (timm)",
    "pretrained": "ImageNet",
    "dataset": "CBIS-DDSM (awsaf49/cbis-ddsm-breast-cancer-image-dataset)",
    "images": "full mammograms (mass + calcification cases)",
    "task": "binary classification (MALIGNANT vs BENIGN)",
    "split": "official train/test, 15% of train held out for validation",
    "loss": "BCEWithLogitsLoss + pos_weight",
    "optimizer": "AdamW",
    "lr": 3e-5,
    "weight_decay": 0.05,
    "scheduler": "CosineAnnealingLR",
    "mixed_precision": true,
    "epochs": 10,
    "batch_size": 16,
    "image_size": 224,
    "threshold": 0.5,
    "seed": 42
}

with open(base_path + "/results/swin_cbis_ddsm_config.json", "w") as f:
    json.dump(config, f, indent=4)

print("Config saved")